In [1]:
%matplotlib inline

import os, random, time, sys
from timeit import default_timer as timer
from datetime import timedelta
from datetime import datetime

import numpy as np
import scipy as sp
from scipy.spatial import distance
from scipy import signal
import pandas as pd
from tqdm import tqdm
import bct

import neurogym as ngym
import torch
import torch.nn as nn

from src.neural_network import RNN, run_testing
from src.utils import normalize_x, build_reg_ken, fix_labels, get_weight_masks, get_file_str

# import plotting libraries
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 10})
plt.rcParams["svg.fonttype"] = "none"
plt.rc('font', family='DejaVu Sans')
import seaborn as sns
sns.set_style("white")
from conn2res import plotting

In [2]:
# directories
datadir = '/home/lindenmp/research_projects/neuro_rnn/data'
modeldir = '/media/lindenmp/storage_ssd/research_projects/neuro_rnn/results/pytorch/model_new'
outdir = '/media/lindenmp/storage_ssd/research_projects/neuro_rnn/results/figs'

# data parameters
task = 'PerceptualDecisionMaking-v0'
dt = 100
batch_size = 128
decision = 500
if task == 'PerceptualDecisionMaking-v0':
    seq_len = 22
seq_len = seq_len + int((decision - 100) / dt)
seq_len = seq_len * 2
print(seq_len)

# RNN model and training parameters
rnn_model = 'rnn-tanh'
hidden_size = 200
n_runs = 25
n_epochs = 10000
lr = 0.001

# regularization parameters
reg_type = 'l2'
reg_weight = 0.001
kernel_type = 'static'
kernel_std_frac = 0.1
comet_buffer_frac = 0.1
comet_tail_frac = 0.25

mask_weights = True

52


In [3]:
if task == 'PerceptualDecisionMaking-v0':
    input_size = 3
    num_classes = 3
elif task == 'MultiSensoryIntegration-v0':
    input_size = 5
    num_classes = 3

timing = {'decision': decision}
env_kwargs = {'dt': dt, 'timing': timing}

config = {
    'datadir': datadir,
    'outdir': outdir,

    # data parameters
    'task': task,
    'dt': dt,
    'seq_len': seq_len,
    'batch_size': batch_size,

    # RNN model and training parameters
    'rnn_model': rnn_model,
    'hidden_size': hidden_size,
    'n_runs': n_runs,
    'n_epochs': n_epochs,
    'lr': lr,
    'mask_weights': mask_weights,

    # regularization parameters
    'reg_type': reg_type,
    'reg_weight': reg_weight,
    'kernel_type': kernel_type,
    'kernel_std_frac': kernel_std_frac,
    'comet_buffer_frac': comet_buffer_frac,
    'comet_tail_frac': comet_tail_frac,

    'env_kwargs': env_kwargs
}

file_str = get_file_str(config)
print(file_str)

task-PerceptualDecisionMaking-v0-52-128-500_model-rnn-tanh-200-25-10000-0.001_wmask-True_reg-l2-0.001-static-0.1


In [4]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# load model
frac = 0.33
masks = get_weight_masks(hidden_size=hidden_size, frac=frac)

n_trials = 50
checkpoint = torch.load(os.path.join(modeldir, file_str + '.pt'), map_location=device)
run = 0
epoch_list = []
N = 10
for idx in range(0, len(list(checkpoint[run].keys())), N):
    epoch_list.append(list(checkpoint[run].keys())[idx])

n_logged_epochs = len(epoch_list)
# n_logged_epochs = len(checkpoint[run].keys())

mask_labels = ['input_output', 'output_input',
               'input_bystanders', 'bystanders_input',
               'output_bystanders', 'bystanders_output']
n_masks = len(mask_labels)
mask_list = []
for i in np.arange(n_masks):
    mask_list.append(masks[mask_labels[i]])
print(n_logged_epochs, n_masks)
print(epoch_list)

cpu
11 6
[0, 999, 1999, 2999, 3999, 4999, 5999, 6999, 7999, 8999, 9999]


In [5]:
for decision in [300, 400, 500, 600]:
    if task == 'PerceptualDecisionMaking-v0':
        seq_len = 22
    seq_len = seq_len + int((decision - 100) / dt)
    seq_len = seq_len * 2
    config['seq_len'] = seq_len

    timing = {'decision': decision}
    env_kwargs = {'dt': dt, 'timing': timing}
    config['env_kwargs'] = env_kwargs

    for reg_type in ['l1', 'l2']:
        config['reg_type'] = reg_type

        for reg_weight in [0.001, 0.01, 0.1]:
            config['reg_weight'] = reg_weight

            for kernel_std_frac in [0.05, 0.1, 0.15]:
                config['kernel_std_frac'] = kernel_std_frac

                for kernel_type in ['static', 'constant']:
                    config['kernel_type'] = kernel_type

                    file_str = get_file_str(config)
                    print(file_str)

                    if os.path.isfile(os.path.join(outdir, '{0}_{1}.png'.format(file_str, 'lesioned-accuracy'))):
                        pass
                    else:
                        # setup dataset
                        dataset = ngym.Dataset(config['task'], env_kwargs=env_kwargs, batch_size=config['batch_size'], seq_len=config['seq_len'])

                        # setup model
                        model = RNN(input_size=input_size, hidden_size=hidden_size, num_classes=num_classes,
                        type=rnn_model, regularization_kernel=np.zeros((hidden_size, hidden_size)),
                        input_weight_mask=masks['input_weight_mask'], output_weight_mask=masks['output_weight_mask']).to(device)
                        model.eval()

                        # load model checkpoint
                        checkpoint = torch.load(os.path.join(modeldir, file_str + '.pt'), map_location=device)
                        # compute accuracy and lesioned accuracy
                        lesioned_accuracy = np.zeros((n_runs, n_logged_epochs, n_masks))
                        accuracy = np.zeros((n_runs, n_logged_epochs))
                        for run in tqdm(np.arange(n_runs)):
                            for i, epoch in enumerate(epoch_list):
                                model.load_state_dict(checkpoint[run][epoch])
                                accuracy[run, i], _, _ = run_testing(dataset=dataset, model=model, n_trials=n_trials, verbose=False)

                                for j, mask in enumerate(mask_list):
                                    model.load_state_dict(checkpoint[run][epoch])
                                    with torch.no_grad():
                                        model.rnn.weight_hh_l0[mask] = 0
                                    lesioned_accuracy[run, i, j], _, _ = run_testing(dataset=dataset, model=model, n_trials=n_trials, verbose=False)

                        # plot
                        f, ax = plt.subplots(2, 1, figsize=(9, 6))
                        x_step = int(n_epochs / (n_logged_epochs-1))
                        color_palette = sns.color_palette("Set2")
                        ax[0].set_ylim([-2.5, 100])
                        ax[0].set_xlabel('Epochs')
                        ax[0].set_ylabel('Test Accuracy (%)')
                        ax[1].set_ylim([-100, 10])
                        ax[1].set_xlabel('Epochs')
                        ax[1].set_ylabel('Test Accuracy (delta, %)')
                        for i in np.arange(n_masks):
                            ax[0].plot(np.arange(0, n_epochs+x_step, x_step), np.median(lesioned_accuracy, axis=0)[:, i] * 100, color=color_palette[i], label=mask_labels[i])
                            ax[1].plot(np.arange(0, n_epochs+x_step, x_step), (np.median(lesioned_accuracy, axis=0)[:, i] - np.median(accuracy, axis=0)) * 100, color=color_palette[i], label=mask_labels[i])
                        ax[0].plot(np.arange(0, n_epochs+x_step, x_step), np.median(accuracy, axis=0) * 100, color='k', label='unlesioned')
                        ax[0].legend(bbox_to_anchor = (1, 1))
                        sns.despine(offset=10, trim=True, left=False, right=True, top=True, bottom=False)

                        f.suptitle(file_str)
                        f.tight_layout()
                        plt.show()
                        f.savefig(os.path.join(outdir, '{0}_{1}.png'.format(file_str, 'lesioned-accuracy')), dpi=300, bbox_inches='tight', pad_inches=0.01)

task-PerceptualDecisionMaking-v0-48-128-300_model-rnn-tanh-200-25-10000-0.001_wmask-True_reg-l1-0.001-static-0.05
task-PerceptualDecisionMaking-v0-48-128-300_model-rnn-tanh-200-25-10000-0.001_wmask-True_reg-l1-0.001-constant-0.05
task-PerceptualDecisionMaking-v0-48-128-300_model-rnn-tanh-200-25-10000-0.001_wmask-True_reg-l1-0.001-static-0.1
task-PerceptualDecisionMaking-v0-48-128-300_model-rnn-tanh-200-25-10000-0.001_wmask-True_reg-l1-0.001-constant-0.1
task-PerceptualDecisionMaking-v0-48-128-300_model-rnn-tanh-200-25-10000-0.001_wmask-True_reg-l1-0.001-static-0.15
task-PerceptualDecisionMaking-v0-48-128-300_model-rnn-tanh-200-25-10000-0.001_wmask-True_reg-l1-0.001-constant-0.15
task-PerceptualDecisionMaking-v0-48-128-300_model-rnn-tanh-200-25-10000-0.001_wmask-True_reg-l1-0.01-static-0.05
task-PerceptualDecisionMaking-v0-48-128-300_model-rnn-tanh-200-25-10000-0.001_wmask-True_reg-l1-0.01-constant-0.05
task-PerceptualDecisionMaking-v0-48-128-300_model-rnn-tanh-200-25-10000-0.001_wmask-